In [ ]:
!pip install -q pyspark optuna

In [ ]:
import os
import glob
import time
import optuna

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
spark = (
    SparkSession.builder
    .appName("TelcoCustomerChurn_Optuna")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark Version:", spark.version)

Spark Version: 3.5.6


In [ ]:
DATA_PATH = "/content/data.xls"

In [ ]:
df = spark.read.csv(
    DATA_PATH,
    header=True,
    inferSchema=True
)

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

df.printSchema()

Number of rows: 7043
Number of columns: 21
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



In [ ]:
df.show(5, truncate=False)

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+-------------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines   |InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract      |PaperlessBilling|PaymentMethod            |MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+-------------------------+--------------+------------+-----+
|7590-VHVEG|Female|0            |Yes    |No        |1     |No          |No phone service|DSL            |No            |Yes         |No              |N

In [ ]:
df = df.withColumn(
    "TotalCharges",
    when(trim(col("TotalCharges")) == "", None)
    .otherwise(col("TotalCharges").cast("double"))
)

df = df.dropna(subset=["TotalCharges"])

print("Rows after cleaning:", df.count())

Rows after cleaning: 7032


In [ ]:
df.groupBy("Churn").count().show()

+-----+-----+
|Churn|count|
+-----+-----+
|   No| 5163|
|  Yes| 1869|
+-----+-----+



In [ ]:
categorical_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

numeric_cols = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

target_col = "Churn"

print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", len(numeric_cols))

Categorical columns: 15
Numerical columns: 4


In [ ]:
train_df, test_df = df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 5690
Testing rows: 1342


In [ ]:
indexers = [
    StringIndexer(
        inputCol=column,
        outputCol=f"{column}_index",
        handleInvalid="keep"
    )
    for column in categorical_cols
]

label_indexer = StringIndexer(
    inputCol=target_col,
    outputCol="label",
    handleInvalid="keep"
)

encoder = OneHotEncoder(
    inputCols=[f"{column}_index" for column in categorical_cols],
    outputCols=[f"{column}_encoded" for column in categorical_cols]
)

assembler_inputs = (
    [f"{column}_encoded" for column in categorical_cols]
    + numeric_cols
)

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

preprocessing_pipeline = Pipeline(
    stages=indexers + [label_indexer, encoder, assembler]
)

In [ ]:
preprocessing_model = preprocessing_pipeline.fit(train_df)

train_processed = preprocessing_model.transform(train_df)
test_processed = preprocessing_model.transform(test_df)

train_processed = train_processed.select("features", "label")
test_processed = test_processed.select("features", "label")

train_processed.cache()
test_processed.cache()

print("Training data:")
train_processed.show(5)

print("Testing data:")
test_processed.show(5)

Training data:
+--------------------+-----+
|            features|label|
+--------------------+-----+
|(45,[0,3,5,6,8,12...|  0.0|
|(45,[1,2,4,6,9,12...|  0.0|
|(45,[1,3,4,6,8,11...|  1.0|
|(45,[0,3,4,6,8,11...|  1.0|
|(45,[0,2,5,6,8,12...|  0.0|
+--------------------+-----+
only showing top 5 rows

Testing data:
+--------------------+-----+
|            features|label|
+--------------------+-----+
|(45,[1,2,4,6,8,11...|  1.0|
|(45,[0,3,4,6,8,11...|  0.0|
|(45,[0,2,4,6,8,12...|  0.0|
|(45,[0,2,4,6,9,11...|  0.0|
|(45,[1,2,4,7,10,1...|  1.0|
+--------------------+-----+
only showing top 5 rows



In [ ]:
rf_default = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42
)

print("Default Random Forest Parameters:")
print(rf_default.extractParamMap())

Default Random Forest Parameters:
{Param(parent='RandomForestClassifier_2568294fe22b', name='seed', doc='random seed.'): 42, Param(parent='RandomForestClassifier_2568294fe22b', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 5, Param(parent='RandomForestClassifier_2568294fe22b', name='maxBins', doc='Max number of bins for discretizing continuous features.  Must be >=2 and >= number of categories for any categorical feature.'): 32, Param(parent='RandomForestClassifier_2568294fe22b', name='minInstancesPerNode', doc='Minimum number of instances each child must have after split. If a split causes the left or right child to have fewer than minInstancesPerNode, the split will be discarded as invalid. Should be >= 1.'): 1, Param(parent='RandomForestClassifier_2568294fe22b', name='minInfoGain', doc='Minimum information gain for a split to be considered at a tree node.'): 0.0, Par

In [ ]:
start_time = time.time()

default_model = rf_default.fit(train_processed)

default_training_time = time.time() - start_time

print(f"Training time: {default_training_time:.2f} seconds")

Training time: 7.70 seconds


In [ ]:
default_predictions = default_model.transform(test_processed)

default_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+--------------------------------------------+
|label|prediction|probability                                 |
+-----+----------+--------------------------------------------+
|1.0  |1.0       |[0.3782726633346055,0.6217273366653945,0.0] |
|0.0  |0.0       |[0.8877393059543156,0.11226069404568433,0.0]|
|0.0  |0.0       |[0.6198303895610885,0.3801696104389115,0.0] |
|0.0  |0.0       |[0.8801255836889108,0.11987441631108917,0.0]|
|1.0  |1.0       |[0.38005840586316414,0.6199415941368358,0.0]|
|0.0  |0.0       |[0.8347461736057056,0.16525382639429445,0.0]|
|0.0  |0.0       |[0.9232264039077354,0.0767735960922647,0.0] |
|0.0  |0.0       |[0.921542900724952,0.07845709927504801,0.0] |
|0.0  |0.0       |[0.893189887688685,0.10681011231131506,0.0] |
|0.0  |0.0       |[0.7904958574281018,0.20950414257189817,0.0]|
+-----+----------+--------------------------------------------+
only showing top 10 rows



In [ ]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

default_accuracy = accuracy_evaluator.evaluate(default_predictions)
default_f1 = f1_evaluator.evaluate(default_predictions)
default_precision = precision_evaluator.evaluate(default_predictions)
default_recall = recall_evaluator.evaluate(default_predictions)

print("===== DEFAULT RANDOM FOREST =====")
print(f"Accuracy : {default_accuracy:.4f}")
print(f"F1 Score : {default_f1:.4f}")
print(f"Precision: {default_precision:.4f}")
print(f"Recall   : {default_recall:.4f}")

===== DEFAULT RANDOM FOREST =====
Accuracy : 0.7951
F1 Score : 0.7722
Precision: 0.7826
Recall   : 0.7951


In [ ]:
optuna_train, validation_df = train_processed.randomSplit(
    [0.8, 0.2],
    seed=42
)

optuna_train.cache()
validation_df.cache()

print("Optuna training rows:", optuna_train.count())
print("Validation rows:", validation_df.count())

Optuna training rows: 4598
Validation rows: 1092


In [ ]:
def objective(trial):

    num_trees = trial.suggest_int(
        "numTrees",
        20,
        100,
        step=20
    )

    max_depth = trial.suggest_int(
        "maxDepth",
        3,
        15
    )

    min_instances = trial.suggest_int(
        "minInstancesPerNode",
        1,
        5
    )

    max_bins = trial.suggest_int(
        "maxBins",
        32,
        128,
        step=32
    )

    feature_subset = trial.suggest_categorical(
        "featureSubsetStrategy",
        ["auto", "sqrt", "log2"]
    )

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=num_trees,
        maxDepth=max_depth,
        minInstancesPerNode=min_instances,
        maxBins=max_bins,
        featureSubsetStrategy=feature_subset,
        seed=42
    )

    model = rf.fit(optuna_train)

    predictions = model.transform(validation_df)

    accuracy = accuracy_evaluator.evaluate(predictions)

    return accuracy

In [ ]:
study = optuna.create_study(
    direction="maximize",
    study_name="RandomForest_Tuning"
)

start_time = time.time()

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

optuna_time = time.time() - start_time

print(f"\nOptuna tuning time: {optuna_time:.2f} seconds")

[I 2026-08-22 06:06:41,573] A new study created in memory with name: RandomForest_Tuning


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-08-22 06:07:06,194] Trial 0 finished with value: 0.7957875457875457 and parameters: {'numTrees': 60, 'maxDepth': 14, 'minInstancesPerNode': 1, 'maxBins': 96, 'featureSubsetStrategy': 'auto'}. Best is trial 0 with value: 0.7957875457875457.
[I 2026-08-22 06:07:16,618] Trial 1 finished with value: 0.8012820512820513 and parameters: {'numTrees': 60, 'maxDepth': 9, 'minInstancesPerNode': 4, 'maxBins': 64, 'featureSubsetStrategy': 'auto'}. Best is trial 1 with value: 0.8012820512820513.
[I 2026-08-22 06:07:19,905] Trial 2 finished with value: 0.7967032967032966 and parameters: {'numTrees': 100, 'maxDepth': 4, 'minInstancesPerNode': 5, 'maxBins': 64, 'featureSubsetStrategy': 'log2'}. Best is trial 1 with value: 0.8012820512820513.
[I 2026-08-22 06:07:24,569] Trial 3 finished with value: 0.8067765567765568 and parameters: {'numTrees': 20, 'maxDepth': 10, 'minInstancesPerNode': 2, 'maxBins': 96, 'featureSubsetStrategy': 'log2'}. Best is trial 3 with value: 0.8067765567765568.
[I 2026-0

In [ ]:
print("===== BEST OPTUNA RESULT =====")

print("Best Validation Accuracy:")
print(f"{study.best_value:.4f}")

print("\nBest Hyperparameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

===== BEST OPTUNA RESULT =====
Best Validation Accuracy:
0.8086

Best Hyperparameters:
numTrees: 40
maxDepth: 15
minInstancesPerNode: 3
maxBins: 32
featureSubsetStrategy: log2


In [ ]:
best_params = study.best_params

rf_optimized = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=best_params["numTrees"],
    maxDepth=best_params["maxDepth"],
    minInstancesPerNode=best_params["minInstancesPerNode"],
    maxBins=best_params["maxBins"],
    featureSubsetStrategy=best_params["featureSubsetStrategy"],
    seed=42
)

print("Optimized Random Forest Parameters:")
print(rf_optimized.extractParamMap())

Optimized Random Forest Parameters:
{Param(parent='RandomForestClassifier_7683ec748b96', name='seed', doc='random seed.'): 42, Param(parent='RandomForestClassifier_7683ec748b96', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 15, Param(parent='RandomForestClassifier_7683ec748b96', name='maxBins', doc='Max number of bins for discretizing continuous features.  Must be >=2 and >= number of categories for any categorical feature.'): 32, Param(parent='RandomForestClassifier_7683ec748b96', name='minInstancesPerNode', doc='Minimum number of instances each child must have after split. If a split causes the left or right child to have fewer than minInstancesPerNode, the split will be discarded as invalid. Should be >= 1.'): 3, Param(parent='RandomForestClassifier_7683ec748b96', name='minInfoGain', doc='Minimum information gain for a split to be considered at a tree node.'): 0.0, 

In [ ]:
start_time = time.time()

optimized_model = rf_optimized.fit(train_processed)

optimized_training_time = time.time() - start_time

print(f"Optimized model training time: {optimized_training_time:.2f} seconds")

Optimized model training time: 13.26 seconds


In [ ]:
optimized_predictions = optimized_model.transform(test_processed)

optimized_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+---------------------------------------------+
|label|prediction|probability                                  |
+-----+----------+---------------------------------------------+
|1.0  |1.0       |[0.4492517571506551,0.5507482428493449,0.0]  |
|0.0  |0.0       |[0.9174652232636159,0.08253477673638403,0.0] |
|0.0  |0.0       |[0.5451347239130436,0.45486527608695637,0.0] |
|0.0  |0.0       |[0.9073283455285235,0.0926716544714764,0.0]  |
|1.0  |1.0       |[0.22736689151313452,0.7726331084868654,0.0] |
|0.0  |0.0       |[0.8132883536128184,0.18671164638718157,0.0] |
|0.0  |0.0       |[0.989139709372683,0.010860290627316912,0.0] |
|0.0  |0.0       |[0.9841763140461474,0.015823685953852622,0.0]|
|0.0  |0.0       |[0.984531844112581,0.015468155887418908,0.0] |
|0.0  |0.0       |[0.8981510926957622,0.10184890730423779,0.0] |
+-----+----------+---------------------------------------------+
only showing top 10 rows



In [ ]:
optimized_accuracy = accuracy_evaluator.evaluate(
    optimized_predictions
)

optimized_f1 = f1_evaluator.evaluate(
    optimized_predictions
)

optimized_precision = precision_evaluator.evaluate(
    optimized_predictions
)

optimized_recall = recall_evaluator.evaluate(
    optimized_predictions
)

print("===== OPTIMIZED RANDOM FOREST =====")
print(f"Accuracy : {optimized_accuracy:.4f}")
print(f"F1 Score : {optimized_f1:.4f}")
print(f"Precision: {optimized_precision:.4f}")
print(f"Recall   : {optimized_recall:.4f}")

===== OPTIMIZED RANDOM FOREST =====
Accuracy : 0.7973
F1 Score : 0.7894
Precision: 0.7870
Recall   : 0.7973


In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "F1 Score",
        "Precision",
        "Recall"
    ],
    "Default Random Forest": [
        default_accuracy,
        default_f1,
        default_precision,
        default_recall
    ],
    "Optimized Random Forest": [
        optimized_accuracy,
        optimized_f1,
        optimized_precision,
        optimized_recall
    ]
})

comparison["Improvement"] = (
    comparison["Optimized Random Forest"]
    - comparison["Default Random Forest"]
)

comparison

,Metric,Default Random Forest,Optimized Random Forest,Improvement
0,Accuracy,0.795082,0.797317,0.002235
1,F1 Score,0.772246,0.789408,0.017162
2,Precision,0.782581,0.786998,0.004417
3,Recall,0.795082,0.797317,0.002235
